# Simulations moteur & correcteurs

### Imports

In [1]:
import numpy as np
from control import tf, c2d, feedback, damp
from control_plotly import pzmap, step, impulse, bode, nichols, rlocus

### Constantes

In [26]:
### mt_ = moteur
mt_L = 1.4e-3 #H
mt_R = 1.4 #Ohm
mt_Km = 57e-3 #Nm/A
mt_J = 2.5e-5 #Kg/m2
mt_f = 1.5e-4 #N.m/(rad/s)

s = tf('s')

#Mt_s = (mt_Km)/((mt_Km**2)+(mt_f+mt_J*s)*(mt_R+mt_L*s))
Mt_s = (0.03353)/(1+s*21e-3)
print(Mt_s)


  0.03353
-----------
0.021 s + 1



### Equations

### Simulations

In [27]:
### Simulation moteur
Mt_s_rpm = (60/(2*np.pi))*Mt_s
tm = np.linspace(start=0,stop=0.5, num = 10000)
step(Mt_s_rpm*1287, tm, x_title="Temps (s)", y_title ="RPM")

In [28]:

Mt_s_H = Mt_s_rpm/(1+Mt_s_rpm)
step([Mt_s_H*158.33,Mt_s_rpm],t=tm, x_title = "Temps (s)", y_title ="Commande")

### Discret

In [29]:
tm_z = np.arange(0, 1, 0.001)
Mt_z_rpm = c2d(Mt_s_rpm, 0.001, 'zoh')
print(Mt_z_rpm)
step([Mt_z_rpm,Mt_s_rpm], t=tm_z)


 0.01489
----------
z - 0.9535

dt = 0.001



In [30]:
Mt_z_H = Mt_z_rpm/(1+Mt_z_rpm)
step(Mt_z_H, t=tm_z)

In [42]:
offset = 1/(10**(42/20)) #+1 for non-oscillating graph
print(offset)
nichols([Mt_z_rpm,Mt_z_rpm/offset])

0.007943282347242814


In [43]:
offset_z = 0.007943282347242814


Mt_z_Hcorr = (Mt_z_rpm/(offset_z))/(1+(Mt_z_rpm/(offset_z)))
tm_z2 = np.arange(0, 0.1, 0.001)
step(Mt_z_Hcorr,t=tm_z2)

In [44]:
tm_z2 = np.arange(0, 5, 0.001)
step([Mt_z_Hcorr,Mt_s_H],t=tm_z2)

### Correcteur

In [57]:
Te = 0.001

Kp = 1/(10**(-40/20))
#1/(offset + 120)
Ki = 10#0.008
Kd = -15#-0.0001#0.09

z = tf('z')

print("Kp : ", Kp)
print("Ki : ", Ki)
print("Kd : ", Kd)

Cz = Kp+Ki*(z/(z-1))+Kd*((z-1)/z)

nichols([Cz*Mt_z_rpm, (Cz*Mt_z_rpm)/(1+Cz*Mt_z_rpm)])

Kp :  100.0
Ki :  10
Kd :  -15


In [58]:
Mt_z_corr = (Cz*Mt_z_rpm)/(1+(Cz*Mt_z_rpm))
tsim = np.arange(0,2,0.01)
step(1*Mt_z_corr*3800, tsim)

In [59]:
Mt_z_errcorr = 1/(1+(Cz*Mt_z_rpm))
step(Mt_z_errcorr*4095, tsim)

In [26]:
alpha_k = 100/(3800*(Kp+Kd+Ki))

step(1*Cz*Mt_z_errcorr*3800*alpha_k, t =np.arange(0,1,0.01), x_title="Temps (s)", y_title="Commande PWM (%)")